<a href="https://colab.research.google.com/github/ahmed-zunaira/APS360_Project/blob/main/notebooks/APS360_Project_Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/ahmed-zunaira/APS360_Project.git

Cloning into 'APS360_Project'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 90 (delta 42), reused 42 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 44.51 MiB | 13.80 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [ ]:
!pip install earthaccess xarray netCDF4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 84.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.6.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.6.0 which is i

In [ ]:
%cd /content/APS360_Project/src

/content/APS360_Project/src


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from torch.utils.data import random_split

from autoencoder_param import Autoencoder
from data_loader import Dataload

In [ ]:
from google.colab import drive
drive.mount ('/content/drive')

Mounted at /content/drive


In [ ]:
# plots the training curve of model
def plot (batch_size=32, learning_rate=0.001, epochs=30):
    train_loss = np.loadtxt("/content/APS360_Project/src/CAE_bs{}_lr{}_epoch{}_train_loss.csv".format(batch_size,learning_rate,epochs))
    val_loss = np.loadtxt("/content/APS360_Project/src/CAE_bs{}_lr{}_epoch{}_val_loss.csv".format(batch_size,learning_rate,epochs))

    plt.title ("Train vs. Validation Loss")
    plt.plot(range(1, len(train_loss)+1), train_loss, label="Train")
    plt.plot(range(1, len(val_loss)+1), val_loss, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")

    plt.legend(loc='best')
    plt.show()

In [ ]:
def evaluate (model, loader, criterion):
    total_loss = 0.0
    model.eval()

    with torch.no_grad():
        for i, data in enumerate(loader, 0):
            img, _ = data
            output = model(img)
            loss = criterion (output, img)

            total_loss += loss.item()

        return total_loss / (i+1)

def train (model, train_data, val_data, batch_size=32, learning_rate=0.001, epochs=30, patience=15):

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    train_loss = np.zeros(epochs)
    val_loss = np.zeros(epochs)

    min_val_loss = float('inf')
    track_bad_epochs = 0
    curr_epochs = epochs

    start_time = time.time()
    for epoch in range(epochs):

        total_loss = 0.0
        model.train()

        for data in train_loader:

            img, _ = data

            output = model(img)
            loss = criterion (output, img)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        train_loss[epoch] = total_loss / len(train_loader)
        val_loss[epoch] = evaluate(model, val_loader, criterion)

        if val_loss[epoch] < min_val_loss:
            min_val_loss = val_loss[epoch]
            track_bad_epochs = 0
            torch.save(model.state_dict(), "CAE_model.pth")
        else:
            track_bad_epochs += 1
            if track_bad_epochs >= patience:
                curr_epochs = epoch+1
                break

    end_time = time.time()
    elapsed_time = end_time - start_time

    train_loss = train_loss[:curr_epochs]
    val_loss = val_loss[:curr_epochs]

    np.savetxt("CAE_bs{}_lr{}_epoch{}_train_loss.csv".format(batch_size,learning_rate,epochs), train_loss)
    np.savetxt("CAE_bs{}_lr{}_epoch{}_val_loss.csv".format(batch_size,learning_rate,epochs), val_loss)

    return min_val_loss

In [ ]:
processed_dir = "/content/drive/MyDrive/Phytoplankton_Project/data/processed"
if not os.path.exists(processed_dir):
    print ("Directory not found")

dataset = Dataload(processed_dir)

train_data, val_data = random_split(dataset, [0.8, 0.2])

In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 21.7 MB/s eta 0:00:00


In [ ]:
import optuna

In [ ]:
def objective(trial):
  # architecture params
  base_channels = trial.suggest_int("base_channels", 8, 32, step=8)
  second_channels = trial.suggest_int("second_channels", 32, 64, step=8)
  third_channels = trial.suggest_int("third_channels", 64, 128, step=8)
  latent_dim = trial.suggest_int("latent_dim", 8, 64, step=8)

  # training_params
  batch_size = trial.suggest_categorical("batch_size", [8, 16, 32, 64])
  learning_rate = trial.suggest_float("learning_rate", 1e-4, 0.1, log=True)
  patience = trial.suggest_int("patience", 3, 25)

  model = Autoencoder(base_channels, second_channels, third_channels, latent_dim)

  return train(model, train_data, val_data, batch_size, learning_rate, epochs=100, patience=patience)

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)

trial = study.best_trial

print('Accuracy: {}'.format(trial.value))
print("Best hyperparameters: {}".format(trial.params))

[I 2026-07-28 10:13:07,967] A new study created in memory with name: no-name-8a18846f-246d-4315-b3ca-516636435f92
[I 2026-07-28 10:38:19,506] Trial 0 finished with value: 0.005455335136502981 and parameters: {'base_channels': 8, 'second_channels': 48, 'third_channels': 96, 'latent_dim': 32, 'batch_size': 32, 'learning_rate': 0.005107376962919825, 'patience': 22}. Best is trial 0 with value: 0.005455335136502981.
[I 2026-07-28 11:09:34,833] Trial 1 finished with value: 0.004274665960110724 and parameters: {'base_channels': 8, 'second_channels': 64, 'third_channels': 128, 'latent_dim': 40, 'batch_size': 16, 'learning_rate': 0.0023036597058007898, 'patience': 25}. Best is trial 1 with value: 0.004274665960110724.
